# Post-hoc F2 evaluation: simulated deployment on 2018-08-01

This notebook presents the F2 evaluation, a post-hoc and exploratory analysis that complements the F1 final evaluation of notebook 09. The hypotheses tested arose from observations made in the F1 test period (Dwork et al., 2015), so F2 neither confirms nor refutes anything about the pre-registered model. The F1 result remains the official one, and F2 only serves to formulate hypotheses to be tested on later data, as set by decision P-06 in `docs/REGISTRO-DE-DECISOES.md` and the plan in `docs/PLANO-F2-POS-HOC.md`.

The notebook does not select a model for production and does not change `production_eligible = false`. Every comparison is read as exploratory, and no difference is called confirmed.

## 1. Post-hoc nature and check of the promise regime

F1 was inconclusive, and three signals were observed in its test period: the promised lead time alone was the best predictor; CatBoost overestimated the risk, because the breach prevalence fell relative to training; and the historical network rates changed sharply. The F2 variants were designed after these observations were known. Any gain in August 2018 may therefore partly reflect fitting to the period itself, and the F2 design tries to contain this risk without eliminating it.

The next cell loads the data exactly as notebook 09 does and checks a necessary condition of the argument: the regime in which the promise discriminates delay did not exist in development. The table shows, by purchase month, the median promised lead time and, only for months before 2018-07-01, the prevalence and the AUC of the negative promised lead time rule. For later months the labels are not read in this check.

In [1]:
from pathlib import Path
import sys

import pandas as pd
import plotly.graph_objects as go
from IPython.display import display

LANG = "en"
project_root = Path.cwd()
while project_root != project_root.parent and not (project_root / "data").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

from ecom_delivery_sla import (
    F1_TRAIN_END,
    F2_EVAL_END,
    F2_EVAL_START,
    F2_UPDATE_DATE,
    build_modeling_frame,
    build_order_features,
    build_supervised_orders,
    describe_run,
    promise_regime_table,
    read_olist_tables,
    run_posthoc_evaluation,
)

POSTHOC_VERSION = "F2"
N_BOOT = 1000
SEED = 42
CACHE_ROOT = project_root / "outputs" / "runs"
F1_MODEL_PATH = project_root / "outputs" / "models" / "catboost_final" / "F1" / "model.joblib"

LABELS = {
    "pt": {
        "ideal": "calibração perfeita",
        "x": "probabilidade média prevista",
        "y": "taxa observada",
        "title": "Curva de confiabilidade em agosto de 2018 (10 bins)",
        "discarded": "Reamostragens descartadas por conter uma só classe: {n} de {total}",
        "post_hoc": "Execução pós-hoc (exploratória): {flag}",
        "eligible": "production_eligible = {value}",
        "reading": "Leitura de todas as comparações: {reading}",
        "update": "Atualização simulada em {date}; treino do F1 com compras anteriores a {train_end}",
        "window": "Avaliação: compras em [{start}, {end})",
        "available": "Pedidos supervisionados disponíveis: {n}",
    },
    "en": {
        "ideal": "perfect calibration",
        "x": "mean predicted probability",
        "y": "observed rate",
        "title": "Reliability curve in August 2018 (10 bins)",
        "discarded": "Resamples discarded for containing a single class: {n} of {total}",
        "post_hoc": "Post-hoc (exploratory) run: {flag}",
        "eligible": "production_eligible = {value}",
        "reading": "Reading of every comparison: {reading}",
        "update": "Simulated update on {date}; F1 training on purchases before {train_end}",
        "window": "Evaluation: purchases in [{start}, {end})",
        "available": "Supervised orders available: {n}",
    },
}[LANG]

tables = read_olist_tables(
    project_root / "data" / "raw" / "olist" / "csv",
    table_names=("orders", "items", "products", "payments", "customers", "sellers", "geolocation"),
)
supervised = build_supervised_orders(tables["orders"])
candidate_features = build_order_features(
    orders=tables["orders"],
    items=tables["items"],
    products=tables["products"],
    payments=tables["payments"],
    customers=tables["customers"],
    sellers=tables["sellers"],
    geolocation=tables["geolocation"],
)
modeling_frame = build_modeling_frame(candidate_features, supervised)
print(LABELS["update"].format(date=F2_UPDATE_DATE, train_end=F1_TRAIN_END))
print(LABELS["window"].format(start=F2_EVAL_START, end=F2_EVAL_END))
print(LABELS["available"].format(n=len(modeling_frame)))

Simulated update on 2018-08-01; F1 training on purchases before 2018-06-01
Evaluation: purchases in [2018-08-01, 2018-09-01)
Supervised orders available: 96470


### Insight

The modeling table holds 96,470 supervised orders, and the F2 windows are applied inside `run_posthoc_evaluation`. The delivery date is used only to decide which labels already existed on the update date and never enters as a feature.

The following table summarizes the behavior of the promise month by month. If the negative promised lead time rule had discriminating power in development, the AUC would depart systematically from chance in the months before July 2018. The following chart shows the median promised lead time as bars and the AUC of the rule as a line, with a chance reference.

In [2]:
regime = promise_regime_table(modeling_frame)
display(regime)

from ecom_delivery_sla_viz import plot_promise_regime

plot_promise_regime(regime, language=LANG).show()

,purchase_month,orders,promise_median,labels_used,breach_prevalence,auc_negative_promise
0,2016-09,1,18.488449,True,1.000000,NaN
1,2016-10,265,54.152095,True,0.007547,0.714829
2,2016-12,1,26.030012,True,0.000000,NaN
3,2017-01,750,38.982905,True,0.029333,0.618007
4,2017-02,1653,30.176470,True,0.029643,0.614586
5,2017-03,2546,22.585220,True,0.045562,0.403697
6,2017-04,2303,25.470544,True,0.065567,0.570412
7,2017-05,3545,23.335255,True,0.029901,0.631309
8,2017-06,3135,23.419803,True,0.030303,0.527798
9,2017-07,3872,23.012668,True,0.027893,0.424702


### Insight

In the development months from 2017-01 to 2018-06, the AUC of the negative promised lead time rule stayed between 0.40 and 0.63 without a trend, that is, close to chance. The median promised lead time ranged between 21.3 and 27.4 days from 2017-03 to 2018-06 and fell to 19.3 days in July and 13.4 days in August 2018. The regime in which the promise discriminates delay therefore does not appear in the development labels, and no variant based on that behavior can be approved by them. This is why F2 is treated as exploratory and why the development gate checks only non-inferiority, not gain. In the chart, months without an AUC have no point on the line because their labels are not read in this check.

## 2. Simulated deployment design and label availability

The simulated operation puts the model into use on July 1, 2018 and updates it on 2018-08-01 with the labels that would already be known on that date. A label is considered known when the order was delivered strictly before the update date (Chapelle, 2014). The evaluation covers the purchases of August 2018, and the confidence interval is obtained by a bootstrap over daily blocks of the purchase date (Künsch, 1989), with 1,000 resamples and seed 42, so that all models are compared on the same resamples. August has only a few weeks, and weekly blocks would produce meaningless intervals.

The internal order of the execution is fixed: first the development gate of variants C, D and E, with no contact with August, and then the August evaluation of the approved variants only. The next cell is the only call in the project to `run_posthoc_evaluation` with `allow_test=True`. The result is frozen in `outputs/runs/final_evaluation/F2/` with the same semantics as F1: on the first execution the result is computed and one line is appended to the ledger, and on any later execution, including in the other language, the stored result is only loaded. No new model is saved to disk.

In [3]:
posthoc, posthoc_meta = run_posthoc_evaluation(
    modeling_frame,
    supervised_orders=supervised,
    allow_test=True,
    version=POSTHOC_VERSION,
    cache_root=CACHE_ROOT,
    f1_model_path=F1_MODEL_PATH,
    n_boot=N_BOOT,
    seed=SEED,
)
print(describe_run(posthoc_meta, language=LANG))
print(LABELS["post_hoc"].format(flag=posthoc_meta["post_hoc"]))

display(posthoc["provenance"])
ledger_path = CACHE_ROOT / "final_evaluation" / "LEDGER.jsonl"
if ledger_path.exists():
    display(pd.read_json(ledger_path, lines=True))

Results loaded from outputs/runs/final_evaluation/F2 (generated on 2026-11-14 12:02, 16.3 min of training; fingerprint a6fd373f)
Post-hoc (exploratory) run: True


,version,post_hoc,reading,production_eligible,created_at,fingerprint,update_date,eval_start,eval_end_exclusive,known_label_orders,eval_orders,eval_blocks,gate_passed,gate_failed,n_boot,seed,f1_model_path,f1_model_sha256
0,F2,True,exploratório,False,2026-11-14T12:02:47,a6fd373f862c7c6a68d92626992dd6bdf08f00b8a26a11...,2018-08-01,2018-08-01,2018-09-01,88097,6351,29,"C,E",D,1000,42,d:\Projetos\E-commerce-Delivery-Network-&-SLA-...,ace91068a027578a40abd579fa968938b332a74c5f484a...


,version,timestamp,fingerprint,train_orders,test_orders,post_hoc,n_boot,seed,gate_passed
0,F1,2026-11-14 10:53:17,ddc0ef97cc7c513c9d44ef32843e1fda7ff118cf7c9445...,77867,12507,False,1000,42,NaN
1,F2,2026-11-14 12:02:47,a6fd373f862c7c6a68d92626992dd6bdf08f00b8a26a11...,88097,6351,True,1000,42,"C,E"


### Insight

The result was computed only once, in the Portuguese run, in 16.3 minutes, and stored with fingerprint `a6fd373f`; the English run only loaded the stored result. The ledger now has two lines: the F1 line, unchanged, and the F2 line, with the `post_hoc` flag set to true and the variants approved by the gate, C and E, recorded. The provenance also reports 88,097 orders with a known label on the update date and 6,351 orders evaluated in 29 daily blocks.

The following table shows, for the purchases of June and July 2018, how many orders had a known label on the update date and what the breach prevalence was among them, compared with the orders still without a label. The following chart stacks the orders with known and unknown labels in each month and shows the prevalence of each group.

In [4]:
display(posthoc["label_availability"])

from ecom_delivery_sla_viz import plot_label_availability

plot_label_availability(posthoc["label_availability"], language=LANG).show()

,purchase_month,as_of,orders,known_labels,share_known,prevalence_known,unknown_labels,prevalence_unknown
0,2018-06,2018-08-01,6096,6070,0.995735,0.008402,26,0.769231
1,2018-07,2018-08-01,6156,4164,0.676413,0.003122,1992,0.097892


### Insight

On the update date, 99.6% of the June 2018 purchases already had a known label, with a prevalence of 0.84%, while the 26 orders not yet delivered had a prevalence of 76.9%. In July, only 67.6% of the labels were known, with a prevalence of 0.31%, against 9.79% among the 1,992 orders still without a label. The slowest orders are precisely those not yet delivered, so the known recent labels are biased towards fast deliveries and towards fewer breaches. This slow-label bias enters the training of every retrained variant and is stronger in the recent window used for the recalibration of variant D. The table is descriptive and does not correct the training. The stacked chart makes the share of still-unknown labels in each month visible.

## 3. Variants A to E

The list of variants is closed, all of them are reported, and none is chosen by the August result. The prevalence, negative promised lead time and logistic regression with lead time and state baselines are refitted on the same known labels as variant B.

| Code | Variant | Mechanism tested |
|---|---|---|
| A | Frozen F1 model, trained on purchases before 2018-06-01 | Reference without update |
| B | Same configuration as F1, retrained on every label known at the update | Monthly retraining |
| C | B with recency weights, with a half-life of 90 days | Adaptation to regime change (Gama et al., 2014) |
| D | B with isotonic recalibration (Zadrozny and Elkan, 2002) fitted only on labels delivered in the last 30 days | Correction of prevalence shift |
| E | B with the `promise_vs_recent_median` feature | Capture of change in the promise policy |

The `promise_vs_recent_median` feature is the ratio between the promised lead time of the order and the median promised lead time of the purchases in the previous 30 days for the same destination state, computed only from strictly earlier purchases, falling back to the global median of the same window. It uses no label and enters only variant E.

The following two tables show the final state of each variant and the size of each set used, with the purchase and delivery dates that bound it.

In [5]:
display(posthoc["variants"])
display(posthoc["populations"])

,code,variant,gated,scored,status
0,A,A_f1_frozen,False,True,avaliada
1,B,B_retrain,False,True,avaliada
2,C,C_recency,True,True,avaliada
3,D,D_recalibration,True,False,reprovada no gate
4,E,E_promise_feature,True,True,avaliada


,variant,role,orders,breaches,prevalence,first_purchase,last_purchase,first_delivery,last_delivery
0,A_f1_frozen,"train (F1, frozen)",77867,5862,0.075282,2016-09-15 12:16:38,2018-05-31 23:51:24,2016-10-11 13:46:32,2018-10-11 16:41:14
1,B_retrain,train (known labels),88097,5922,0.067221,2016-09-15 12:16:38,2018-07-30 13:26:53,2016-10-11 13:46:32,2018-07-31 23:44:41
2,C_recency,train (known labels),88097,5922,0.067221,2016-09-15 12:16:38,2018-07-30 13:26:53,2016-10-11 13:46:32,2018-07-31 23:44:41
3,E_promise_feature,train (known labels),88097,5922,0.067221,2016-09-15 12:16:38,2018-07-30 13:26:53,2016-10-11 13:46:32,2018-07-31 23:44:41
4,baselines,train (known labels),88097,5922,0.067221,2016-09-15 12:16:38,2018-07-30 13:26:53,2016-10-11 13:46:32,2018-07-31 23:44:41
5,all,evaluation (August purchases),6351,393,0.061880,2018-08-01 00:02:17,2018-08-29 15:00:37,2018-08-02 15:42:36,2018-09-25 00:47:25


### Insight

Variants A, B, C and E were scored in August, and variant D, which failed the gate, was not evaluated. The retrained variants and the baselines were fitted on 88,097 orders with a known label, with a prevalence of 6.72% and the last delivery on 2018-07-31, against 77,867 orders and a prevalence of 7.53% in the frozen F1 training set; the drop in prevalence reflects the inclusion of the recent fast-delivery labels. The evaluated population has 6,351 August purchases, with 393 breaches and a prevalence of 6.19%. No label delivered on or after the update date is in the training sets, and variant A, without retraining, serves as the reference to measure the effect of the update.

## 4. Development gate

Before any contact with August, each mechanism of variants C, D and E is applied in the development folds, from 2017-10 to 2018-06 with a 30-day embargo, and compared with the F1 configuration in the same fold. In the folds, the update date is the end of the training window. The age used in the weights of variant C is relative to that date, and the recalibration window of variant D is the last 30 days of training purchases, because the folds do not use the delivery date as a criterion. This is an adaptation of the real mechanism, which uses the delivery date, and must be read as such.

A variant proceeds to the August evaluation only if it is non-inferior in development: the mean AP must be at least 95% of the mean AP of the F1 configuration, and the mean ECE must be at most 0.01 above the mean ECE of the F1 configuration. Failed variants appear in the final table as failed at the gate and are not scored in August.

The first table summarizes the gate by variant, with the means, the AP ratio, the ECE difference and the decision. The second gives the metrics by fold, including the F1 configuration used as the reference. The following chart shows the AP of each variant by fold, with the F1 configuration dashed.

In [6]:
display(posthoc["gate_summary"])
display(posthoc["gate_folds"])

from ecom_delivery_sla_viz import plot_metric_by_fold

plot_metric_by_fold(
    posthoc["gate_folds"],
    fold_col="fold",
    model_col="variant",
    metric_col="average_precision",
    reference="F1_config",
    language=LANG,
).show()

,variant,n_folds,mean_ap,reference_mean_ap,ap_ratio,ap_ratio_threshold,passes_ap,mean_ece,reference_mean_ece,ece_difference,ece_margin,passes_ece,passed,status
0,C_recency,9,0.170719,0.174999,0.975542,0.95,True,0.058680,0.057619,0.001061,0.01,True,True,avaliada
1,D_recalibration,9,0.145307,0.174999,0.830333,0.95,False,0.065182,0.057619,0.007563,0.01,True,False,reprovada no gate
2,E_promise_feature,9,0.169836,0.174999,0.970496,0.95,True,0.057674,0.057619,0.000056,0.01,True,True,avaliada


,fold,variant,update_date,n_validation,n_fit,n_calibration,average_precision,ece_10bins,brier
0,2017-10,F1_config,2017-09-01,4478,22264,0,0.084222,0.012413,0.039591
1,2017-10,C_recency,2017-09-01,4478,22264,0,0.078871,0.017704,0.039853
2,2017-10,D_recalibration,2017-09-01,4478,18232,4032,0.062233,0.013493,0.039909
3,2017-10,E_promise_feature,2017-09-01,4478,22264,0,0.075715,0.015709,0.039834
4,2017-11,F1_config,2017-10-02,7288,26538,0,0.207368,0.087310,0.113753
5,2017-11,C_recency,2017-10-02,7288,26538,0,0.209523,0.092434,0.114730
6,2017-11,D_recalibration,2017-10-02,7288,22417,4121,0.172526,0.081899,0.113618
7,2017-11,E_promise_feature,2017-10-02,7288,26538,0,0.186419,0.088827,0.114706
8,2017-12,F1_config,2017-11-01,5513,30892,0,0.122211,0.043552,0.069908
9,2017-12,C_recency,2017-11-01,5513,30892,0,0.120922,0.046552,0.070176


### Insight

Variants C and E were approved, with an AP ratio of 0.976 and 0.970 relative to the F1 configuration and an ECE difference of +0.001 and +0.0001. Variant D failed on AP, with a ratio of 0.830 against the 0.95 threshold, although its ECE difference of +0.008 was within the margin, and its AP was below the reference in all nine folds. The gate measures non-inferiority, not gain: C and E were approved for not harming development, not for demonstrating benefit. Because the recent window in the folds is defined by purchase date, without censoring by delivery date, the gate does not reproduce the slow-label bias observed in the real update. The failed variant is not re-evaluated with other thresholds after the August result is known. The by-fold chart allows each variant to be compared with the F1 configuration in every fold.

## 5. August evaluation with daily block intervals

The classification metrics are average precision (primary), ROC AUC, Brier, ECE in ten bins, and precision and recall in the top 5% and 10% of risk, all with the 95% interval of the daily block bootstrap. For each evaluated variant, the paired AP difference is computed against the logistic regression with lead time and state and against the negative promised lead time rule, with its 95% interval. No confirmed or not confirmed label is applied: the table only shows the sign of the difference, whether the interval excludes zero, and the fixed exploratory reading. Brier and ECE are not defined for the negative promised lead time rule, which is not a probability.

The following table gives the point estimate on the full sample and the interval of each metric, for the evaluated variants and for the baselines, and the status of each variant.

In [7]:
metrics = posthoc["classification_metrics"]
display(metrics[["model", "metric", "estimate", "ci_low", "ci_high", "status"]])
sampled = metrics.dropna(subset=["n_boot"])
print(LABELS["discarded"].format(n=int(sampled["n_discarded"].iloc[0]), total=int(sampled["n_boot"].iloc[0])))

,model,metric,estimate,ci_low,ci_high,status
0,A_f1_frozen,average_precision,0.069026,0.054170,0.083777,avaliada
1,A_f1_frozen,roc_auc,0.555992,0.516885,0.599745,avaliada
2,A_f1_frozen,brier,0.063841,0.048610,0.079037,avaliada
3,A_f1_frozen,ece_10bins,0.041298,0.034067,0.056659,avaliada
4,A_f1_frozen,precision_at_5pct,0.081761,0.053690,0.107878,avaliada
5,A_f1_frozen,recall_at_5pct,0.066158,0.042040,0.096222,avaliada
6,A_f1_frozen,precision_at_10pct,0.066038,0.046233,0.088289,avaliada
7,A_f1_frozen,recall_at_10pct,0.106870,0.073115,0.157310,avaliada
8,B_retrain,average_precision,0.068454,0.053698,0.082890,avaliada
9,B_retrain,roc_auc,0.556883,0.517592,0.600773,avaliada


Resamples discarded for containing a single class: 0 of 1000


### Insight

None of the evaluated variants stood out in August. AP was 0.069 for variant A (95% CI from 0.054 to 0.084), 0.068 for B, 0.067 for C and 0.069 for E, and ROC AUC stayed between 0.542 and 0.565, slightly above the prevalence baseline, whose AP is 0.062. The logistic regression with promise and state reached an AP of 0.094 (CI from 0.071 to 0.121) and an AUC of 0.601, and the negative promised lead time rule reached an AP of 0.179 (CI from 0.105 to 0.284) and an AUC of 0.777. The comparison between A and B, which isolates the effect of updating the model, shows no relevant difference, and the promise feature of variant E barely changed the result of B. No resample was discarded, and the intervals reflect the 29 days of the period and the correlation within each day.

The reliability curve compares, in ten bins, the mean predicted probability with the observed breach rate for the models that output probabilities. The table of precision and recall in the top 5% and 10% of risk summarizes the use of the score for prioritization.

In [8]:
reliability = posthoc["reliability"]
display(reliability)

figure = go.Figure()
for model, part in reliability.groupby("model", sort=False):
    figure.add_trace(
        go.Scatter(x=part["mean_predicted"], y=part["observed_rate"], mode="lines+markers", name=model)
    )
figure.add_trace(
    go.Scatter(x=[0, 1], y=[0, 1], mode="lines", line={"dash": "dash"}, name=LABELS["ideal"])
)
figure.update_layout(title=LABELS["title"], xaxis_title=LABELS["x"], yaxis_title=LABELS["y"])
figure.show()

display(posthoc["top_k"])

,model,bin,bin_lower,bin_upper,n,mean_predicted,observed_rate
0,A_f1_frozen,1,0.0,0.1,4521,0.053573,0.061712
1,A_f1_frozen,2,0.1,0.2,1213,0.139892,0.060181
2,A_f1_frozen,3,0.2,0.3,461,0.242664,0.060738
3,A_f1_frozen,4,0.3,0.4,123,0.339002,0.089431
4,A_f1_frozen,5,0.4,0.5,24,0.431044,0.083333
5,A_f1_frozen,6,0.5,0.6,2,0.558429,0.000000
6,A_f1_frozen,7,0.6,0.7,4,0.633337,0.000000
7,A_f1_frozen,8,0.7,0.8,2,0.718727,0.000000
8,A_f1_frozen,9,0.8,0.9,1,0.800651,0.000000
9,A_f1_frozen,10,0.9,1.0,0,NaN,NaN


,model,top_fraction,precision,precision_ci_low,precision_ci_high,recall,recall_ci_low,recall_ci_high
0,A_f1_frozen,0.05,0.081761,0.053690,0.107878,0.066158,0.042040,0.096222
1,A_f1_frozen,0.10,0.066038,0.046233,0.088289,0.106870,0.073115,0.157310
2,B_retrain,0.05,0.075472,0.041973,0.101855,0.061069,0.033043,0.091252
3,B_retrain,0.10,0.064465,0.043829,0.088381,0.104326,0.073491,0.149655
4,C_recency,0.05,0.072327,0.040397,0.101984,0.058524,0.034085,0.085724
5,C_recency,0.10,0.069182,0.042699,0.092784,0.111959,0.072577,0.156874
6,E_promise_feature,0.05,0.075472,0.042207,0.103787,0.061069,0.032035,0.098046
7,E_promise_feature,0.10,0.067610,0.044630,0.089724,0.109415,0.070030,0.158787
8,baseline_prevalence,0.05,0.062893,0.010134,0.141540,0.050891,0.009091,0.111671
9,baseline_prevalence,0.10,0.056604,0.018122,0.117559,0.091603,0.032150,0.181626


### Insight

The calibration error changed direction with the update without shrinking. Variant A overestimates risk: in the bins from 0.1 to 0.5, the mean predicted probability goes from 0.14 to 0.43, while the observed rate stays between 0.06 and 0.09. The retrained variants underestimate: B, C and E place, respectively, 6,302, 6,346 and 6,298 of the 6,351 orders in the bin below 0.1, with a mean predicted probability of 0.021, 0.013 and 0.023, against an observed rate of 0.062. ECE stayed between 0.039 and 0.049 in the four variants, against 0.005 for the prevalence baseline, whose training rate was close to the August rate. The underestimation by the retrained models is compatible with the slow-label bias shown in section 2, but this explanation was not tested. Variant D, which would target this deviation directly, was not evaluated because it failed the gate. Bins with few orders must be read with caution.

The following table gives the paired AP difference of each variant against the two references, with the 95% interval obtained on the same daily resamples. The interval chart brings together the AP of each variant and of the baselines, with the 95% CI, and the paired differences against the two references, with the zero line.

In [9]:
paired = posthoc["paired_difference"]
display(
    paired[
        [
            "model",
            "reference",
            "estimate_model",
            "estimate_reference",
            "difference",
            "ci_low",
            "ci_high",
            "difference_sign",
            "ci_excludes_zero",
            "reading",
            "status",
        ]
    ]
)

from ecom_delivery_sla_viz import plot_metric_intervals

plot_metric_intervals(metrics, metric="average_precision", paired=paired, language=LANG).show()

,model,reference,estimate_model,estimate_reference,difference,ci_low,ci_high,difference_sign,ci_excludes_zero,reading,status
0,A_f1_frozen,baseline_logistic_promise_plus_state,0.069026,0.093841,-0.024816,-0.043786,-0.010757,negative,True,exploratório,avaliada
1,A_f1_frozen,baseline_negative_promised_lead_days,0.069026,0.178748,-0.109723,-0.208806,-0.039556,negative,True,exploratório,avaliada
2,B_retrain,baseline_logistic_promise_plus_state,0.068454,0.093841,-0.025388,-0.044839,-0.009873,negative,True,exploratório,avaliada
3,B_retrain,baseline_negative_promised_lead_days,0.068454,0.178748,-0.110295,-0.209815,-0.040177,negative,True,exploratório,avaliada
4,C_recency,baseline_logistic_promise_plus_state,0.067454,0.093841,-0.026387,-0.046548,-0.011402,negative,True,exploratório,avaliada
5,C_recency,baseline_negative_promised_lead_days,0.067454,0.178748,-0.111294,-0.209972,-0.041853,negative,True,exploratório,avaliada
6,E_promise_feature,baseline_logistic_promise_plus_state,0.069296,0.093841,-0.024545,-0.043787,-0.009941,negative,True,exploratório,avaliada
7,E_promise_feature,baseline_negative_promised_lead_days,0.069296,0.178748,-0.109452,-0.209801,-0.039473,negative,True,exploratório,avaliada
8,D_recalibration,baseline_logistic_promise_plus_state,NaN,NaN,NaN,NaN,NaN,not evaluated,False,exploratório,reprovada no gate
9,D_recalibration,baseline_negative_promised_lead_days,NaN,NaN,NaN,NaN,NaN,not evaluated,False,exploratório,reprovada no gate


### Insight

Against the logistic regression with promise and state, the paired AP difference stayed between −0.026 and −0.025 in the four variants, with a 95% CI entirely below zero (for variant A, from −0.044 to −0.011). Against the negative promised lead time rule, the difference was around −0.11, also with a CI below zero (for variant A, from −0.209 to −0.040). With several variants compared against two references and no adjustment for multiple comparisons (Benjamini and Hochberg, 1995), these differences are described as exploratory effect sizes, not as confirmed results. The sign is nevertheless the same in every comparison: none of the tested update mechanisms brought CatBoost closer to the promise-based references. In the chart, the zero line separates the differences in favor of from those against each reference.

## 6. Exploratory reading and limitations

The reading of every result in this notebook is exploratory. F1 remains the official result, and F2 is a complementary analysis. A variant can only be recommended for a next cycle, to be validated on data after August 2018, and not for production. The next cell repeats the fixed reading and records the `production_eligible` field.

In [10]:
print(LABELS["reading"].format(reading=", ".join(sorted(paired["reading"].unique()))))
production_eligible = False
print(LABELS["eligible"].format(value=production_eligible))

Reading of every comparison: exploratório
production_eligible = False


### Insight

The exploratory reading of F2 is that none of the tested update mechanisms recovered, in August 2018, the performance of the references based on the promised lead time. The explanation compatible with the tables, which was not tested, combines three factors: the regime in which the promise discriminates delay does not exist in the training labels, the recent labels available at the update are biased towards fast deliveries, and the development gate does not reproduce that bias. The negative promised lead time rule prevails because it has a fixed sign and does not depend on learning that relationship. Two hypotheses are recorded for a next cycle, to be validated on data after August 2018: training the update only on cohorts with mature outcomes, or treating in-transit orders as censored (Kaplan and Meier, 1958), and using the promise rule as an alert of policy change rather than as a model.

Four limitations must accompany any citation of F2. First, August 2018 was already observed in F1, and the variants were designed after that observation, so the period is not an independent holdout. Second, several variants are compared against two references without correction for multiple comparisons. Third, the evaluated period has 29 daily blocks and a small population, which results in wide intervals. Fourth, the dataset has no data after August 2018, so no hypothesis can be tested outside the period already consulted; July and August 2018, consulted in F1 and F2, are exhausted for new evaluations.

For these reasons, `production_eligible` remains `false`, as set by the governance rule of the register, and no variant is recommended for operational use.

## References

Benjamini, Y. and Hochberg, Y. (1995). Controlling the false discovery rate: a practical and powerful approach to multiple testing. *Journal of the Royal Statistical Society, Series B*, 57(1), 289–300. https://doi.org/10.1111/j.2517-6161.1995.tb02031.x

Chapelle, O. (2014). Modeling delayed feedback in display advertising. *Proceedings of the 20th ACM SIGKDD International Conference on Knowledge Discovery and Data Mining*, 1097–1105. https://doi.org/10.1145/2623330.2623634

Dwork, C., Feldman, V., Hardt, M., Pitassi, T., Reingold, O., and Roth, A. (2015). The reusable holdout: preserving validity in adaptive data analysis. *Science*, 349(6248), 636–638. https://doi.org/10.1126/science.aaa9375

Gama, J., Žliobaitė, I., Bifet, A., Pechenizkiy, M., and Bouchachia, A. (2014). A survey on concept drift adaptation. *ACM Computing Surveys*, 46(4), article 44. https://doi.org/10.1145/2523813

Kaplan, E. L. and Meier, P. (1958). Nonparametric estimation from incomplete observations. *Journal of the American Statistical Association*, 53(282), 457–481. https://doi.org/10.1080/01621459.1958.10501452

Künsch, H. R. (1989). The jackknife and the bootstrap for general stationary observations. *Annals of Statistics*, 17(3), 1217–1241. https://doi.org/10.1214/aos/1176347265

Zadrozny, B. and Elkan, C. (2002). Transforming classifier scores into accurate multiclass probability estimates. *Proceedings of the 8th ACM SIGKDD International Conference on Knowledge Discovery and Data Mining*, 694–699. https://doi.org/10.1145/775047.775151